In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
)

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

train = pd.read_csv("shared_data/train_clean.csv")
val = pd.read_csv("shared_data/val_clean.csv")
test = pd.read_csv("shared_data/test_clean.csv")
target = "is_canceled"

X_train, y_train = train.drop(columns=[target]), train[target]
X_val, y_val = val.drop(columns=[target]), val[target]
X_test, y_test = test.drop(columns=[target]), test[target]

assert X_train.columns.equals(X_val.columns)
assert X_train.columns.equals(X_test.columns)
assert X_train.shape[1] == 24, "RQ1 A must contain all 24 shared features."
for features, labels in [(X_train, y_train), (X_val, y_val), (X_test, y_test)]:
    assert not features.isna().any().any()
    assert set(labels.unique()) == {0, 1}

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)
print("Data validation: PASSED")

depths = [3, 5, 8, 10, 15, 20]
LEAF_SIZES = [1, 10, 50]
THRESHOLDS = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]
RANDOM_STATE = 42

In [ ]:
def classification_metrics(labels, prediction):
    return {
        "Accuracy": accuracy_score(labels, prediction),
        "Precision": precision_score(labels, prediction, zero_division=0),
        "Recall": recall_score(labels, prediction, zero_division=0),
        "F1": f1_score(labels, prediction, zero_division=0),
    }


def probability_metrics(labels, probabilities, threshold):
    prediction = (probabilities >= threshold).astype(int)
    return {
        **classification_metrics(labels, prediction),
        "ROC_AUC": roc_auc_score(labels, probabilities),
    }


def canceled_probabilities(model, encoded_features):
    positive_index = list(model.classes_).index(1)
    return model.predict_proba(encoded_features)[:, positive_index]


def fit_decision_tree(Xtr, ytr, Xva, yva, output_prefix):
    """Fit on train; choose (depth, leaf size), then threshold on validation F1."""
    assert Xtr.columns.equals(Xva.columns)
    assert set(ytr.unique()) == {0, 1}
    categorical_cols = Xtr.select_dtypes(exclude="number").columns.tolist()
    preprocessor = ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)],
        remainder="passthrough",
    )
    Xtr_encoded = preprocessor.fit_transform(Xtr)
    Xva_encoded = preprocessor.transform(Xva)

    candidates, tuning_rows = {}, []
    for depth in depths:
        for leaf_size in LEAF_SIZES:
            candidate = DecisionTreeClassifier(
                criterion="entropy", max_depth=depth,
                min_samples_leaf=leaf_size, random_state=RANDOM_STATE,
            )
            candidate.fit(Xtr_encoded, ytr)
            candidates[(depth, leaf_size)] = candidate
            train_prediction = candidate.predict(Xtr_encoded)
            val_prediction = candidate.predict(Xva_encoded)
            train_f1 = f1_score(ytr, train_prediction, zero_division=0)
            val_f1 = f1_score(yva, val_prediction, zero_division=0)
            tuning_rows.append({
                "max_depth": depth, "min_samples_leaf": leaf_size,
                "train_accuracy": accuracy_score(ytr, train_prediction),
                "train_f1": train_f1,
                "val_accuracy": accuracy_score(yva, val_prediction),
                "val_precision": precision_score(yva, val_prediction, zero_division=0),
                "val_recall": recall_score(yva, val_prediction, zero_division=0),
                "val_f1": val_f1, "train_val_f1_gap": train_f1 - val_f1,
                "tree_leaves": candidate.get_n_leaves(),
            })

    depth_results = pd.DataFrame(tuning_rows)
    # idxmax returns the first maximum in the specified grid order.
    best_row = depth_results.loc[depth_results["val_f1"].idxmax()]
    best_depth = int(best_row["max_depth"])
    best_leaf_size = int(best_row["min_samples_leaf"])
    model = candidates[(best_depth, best_leaf_size)]
    original_grid = depth_results.loc[depth_results["min_samples_leaf"] == 1]
    original_row = original_grid.loc[original_grid["val_f1"].idxmax()]
    regularization_comparison = pd.DataFrame([
        {"Selection": "Original depth grid (min_samples_leaf=1)", **original_row.to_dict()},
        {"Selection": "Expanded depth and leaf-size grid", **best_row.to_dict()},
    ])
    for column in ["max_depth", "min_samples_leaf", "tree_leaves"]:
        regularization_comparison[column] = regularization_comparison[column].astype(int)
    regularization_comparison["Prediction_rule"] = "Default predict, before threshold selection"

    val_prob = canceled_probabilities(model, Xva_encoded)
    threshold_results = pd.DataFrame([
        {"Threshold": threshold,
         **classification_metrics(yva, (val_prob >= threshold).astype(int))}
        for threshold in THRESHOLDS
    ])
    threshold = float(threshold_results.loc[threshold_results["F1"].idxmax(), "Threshold"])

    # Retain the old filename, now with both tuning dimensions explicitly recorded.
    depth_results.to_csv(output_dir / f"{output_prefix}_depth_tuning.csv", index=False)
    threshold_results.to_csv(output_dir / f"{output_prefix}_threshold_tuning.csv", index=False)
    regularization_comparison.to_csv(
        output_dir / f"{output_prefix}_regularization_comparison.csv", index=False
    )
    return {
        "preprocessor": preprocessor, "model": model,
        "max_depth": best_depth, "min_samples_leaf": best_leaf_size,
        "threshold": threshold, "depth_results": depth_results,
        "threshold_results": threshold_results,
        "regularization_comparison": regularization_comparison,
    }


def predict_probabilities(experiment, features):
    encoded = experiment["preprocessor"].transform(features)
    return canceled_probabilities(experiment["model"], encoded)

In [ ]:
main_experiment = fit_decision_tree(X_train, y_train, X_val, y_val, "decision_tree")
preprocessor = main_experiment["preprocessor"]
best_dt = main_experiment["model"]
best_depth = main_experiment["max_depth"]
best_leaf_size = main_experiment["min_samples_leaf"]
final_threshold = main_experiment["threshold"]
depth_results = main_experiment["depth_results"]
threshold_results = main_experiment["threshold_results"]

print("Selected max_depth:", best_depth)
print("Selected min_samples_leaf:", best_leaf_size)
print("Selected threshold:", final_threshold)
print("Validation depth/leaf-size tuning (default predict rule):")
display(depth_results.round(4))
print("Validation-selected original and expanded grids (before threshold selection):")
display(main_experiment["regularization_comparison"].round(4))
print("Validation threshold tuning (selected tree):")
display(threshold_results.round(4))

In [ ]:
# These plots describe validation model selection, not final test performance.
# Each leaf size has its own panel so different configurations are never joined.
fig, axes = plt.subplots(1, len(LEAF_SIZES), figsize=(15, 4), sharey=True)
for ax, leaf_size in zip(axes, LEAF_SIZES):
    subset = depth_results.loc[depth_results["min_samples_leaf"] == leaf_size]
    ax.plot(subset["max_depth"], subset["train_f1"], marker="o", label="Training F1")
    ax.plot(subset["max_depth"], subset["val_f1"], marker="o", label="Validation F1")
    ax.set(xlabel="Maximum tree depth", title=f"min_samples_leaf = {leaf_size}")
    ax.set_xticks(depths)
    ax.legend()
    ax.grid(alpha=0.3)
axes[0].set_ylabel("F1 score")
fig.suptitle("Decision tree: training and validation F1 (default predict)")
fig.tight_layout()
fig.savefig(output_dir / "decision_tree_depth_tuning.png", dpi=200, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
for metric in ["Precision", "Recall", "F1"]:
    ax.plot(threshold_results["Threshold"], threshold_results[metric], marker="o", label=metric)
ax.set(xlabel="Classification threshold", ylabel="Score", title="Decision tree: validation threshold selection")
ax.set_xticks(THRESHOLDS)
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(output_dir / "decision_tree_threshold_tuning.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
test_prob = predict_probabilities(main_experiment, X_test)
test_pred = (test_prob >= final_threshold).astype(int)
final_results = pd.DataFrame([{
    "Model": "Decision Tree",
    "max_depth": best_depth, "min_samples_leaf": best_leaf_size,
    "Threshold": final_threshold,
    **probability_metrics(y_test, test_prob, final_threshold),
}])
final_results.to_csv(output_dir / "decision_tree_final_results.csv", index=False)
print("Final Decision Tree test results:")
display(final_results.round(4))

pd.DataFrame([{
    "Model": "Decision Tree", "criterion": "entropy", "max_depth": best_depth,
    "min_samples_leaf": best_leaf_size,
    "Threshold": final_threshold, "Selection_metric": "Validation F1",
    "Train_rows": len(X_train), "Validation_rows": len(X_val), "Test_rows": len(X_test),
    "n_features": X_train.shape[1], "random_state": RANDOM_STATE,
}]).to_csv(output_dir / "decision_tree_configuration.csv", index=False)

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_test_pred = baseline.predict(X_test)
baseline_test_prob = canceled_probabilities(baseline, X_test)
test_comparison = pd.DataFrame([
    {"Model": "Majority Class Baseline",
     **classification_metrics(y_test, baseline_test_pred),
     "ROC_AUC": roc_auc_score(y_test, baseline_test_prob)},
    {"Model": "Decision Tree", **probability_metrics(y_test, test_prob, final_threshold)},
])
test_comparison.to_csv(output_dir / "decision_tree_vs_baseline.csv", index=False)
display(test_comparison.round(4))

In [ ]:
history_cols = [
    "is_repeated_guest", "previous_cancellations", "previous_bookings_not_canceled"
]
X_train_no_history = X_train.drop(columns=history_cols)
X_val_no_history = X_val.drop(columns=history_cols)
X_test_no_history = X_test.drop(columns=history_cols)
assert X_train_no_history.shape[1] == 21

no_history_experiment = fit_decision_tree(
    X_train_no_history, y_train, X_val_no_history, y_val, "decision_tree_rq1_b"
)
no_history_test_prob = predict_probabilities(no_history_experiment, X_test_no_history)

rq1_rows = []
for setting, feature_count, experiment, probabilities in [
    ("A: all 24 features", 24, main_experiment, test_prob),
    ("B: without booking history", 21, no_history_experiment, no_history_test_prob),
]:
    rq1_rows.append({
        "Setting": setting, "n_features": feature_count,
        "max_depth": experiment["max_depth"],
        "min_samples_leaf": experiment["min_samples_leaf"],
        "Threshold": experiment["threshold"],
        "n_test": len(y_test), "Cancel_rate": y_test.mean(),
        **probability_metrics(y_test, probabilities, experiment["threshold"]),
    })
rq1_test_results = pd.DataFrame(rq1_rows)
rq1_test_results.to_csv(output_dir / "decision_tree_rq1_results.csv", index=False)
print("RQ1 final test comparison:")
display(rq1_test_results.round(4))

rq1_baseline_comparison = pd.DataFrame([
    {"Model": "Majority Class Baseline (shared by A and B)",
     "Setting": "Shared training labels and test rows", "n_test": len(y_test),
     "Cancel_rate": y_test.mean(),
     **classification_metrics(y_test, baseline_test_pred),
     "ROC_AUC": roc_auc_score(y_test, baseline_test_prob)},
    *[{"Model": "Decision Tree", **row} for row in rq1_rows],
])
rq1_baseline_comparison.to_csv(output_dir / "decision_tree_rq1_vs_baseline.csv", index=False)
print("RQ1 comparison with the reused main baseline:")
display(rq1_baseline_comparison.round(4))

# Traceable predictions for independently recalculating every RQ1 metric.
rq1_prediction_rows = []
for setting, experiment, probabilities in [
    ("A", main_experiment, test_prob),
    ("B", no_history_experiment, no_history_test_prob),
]:
    rq1_prediction_rows.append(pd.DataFrame({
        "Setting": setting, "test_row_id": np.arange(len(y_test)),
        "actual": y_test.to_numpy(), "probability": probabilities,
        "predicted": (probabilities >= experiment["threshold"]).astype(int),
        "Threshold": experiment["threshold"],
    }))
rq1_predictions = pd.concat(rq1_prediction_rows, ignore_index=True)
rq1_predictions.to_csv(output_dir / "decision_tree_rq1_predictions.csv", index=False)


In [ ]:
hotels = ["City Hotel", "Resort Hotel"]
assert set(train["hotel"].unique()) == set(hotels)


def split_by_hotel(frame, hotel_name):
    subset = frame.loc[frame["hotel"] == hotel_name]
    assert len(subset) > 0
    return subset.drop(columns=[target, "hotel"]), subset[target]


rq2_rows, rq2_baseline_rows, rq2_prediction_rows = [], [], []
rq2_experiments = {}
for source_hotel in hotels:
    X_source_train, y_source_train = split_by_hotel(train, source_hotel)
    X_source_val, y_source_val = split_by_hotel(val, source_hotel)
    source_slug = source_hotel.lower().replace(" ", "_")
    source_experiment = fit_decision_tree(
        X_source_train, y_source_train, X_source_val, y_source_val,
        f"decision_tree_rq2_{source_slug}",
    )
    rq2_experiments[source_hotel] = source_experiment
    source_baseline = DummyClassifier(strategy="most_frequent")
    source_baseline.fit(X_source_train, y_source_train)
    print(f"{source_hotel}: depth={source_experiment['max_depth']}, "
          f"min_samples_leaf={source_experiment['min_samples_leaf']}, "
          f"threshold={source_experiment['threshold']:.2f}")

    for evaluation_hotel in hotels:
        X_hotel_test, y_hotel_test = split_by_hotel(test, evaluation_hotel)
        hotel_test_prob = predict_probabilities(source_experiment, X_hotel_test)
        shared_metadata = {
            "Train_hotel": source_hotel, "Test_hotel": evaluation_hotel,
            "Type": "in-hotel" if source_hotel == evaluation_hotel else "transfer",
            "n_train": len(y_source_train), "n_val": len(y_source_val),
            "n_test": len(y_hotel_test), "Cancel_rate": y_hotel_test.mean(),
        }
        baseline_prediction = source_baseline.predict(X_hotel_test)
        baseline_probability = canceled_probabilities(source_baseline, X_hotel_test)
        baseline_metrics = {
            **classification_metrics(y_hotel_test, baseline_prediction),
            "ROC_AUC": roc_auc_score(y_hotel_test, baseline_probability),
        }
        rq2_prediction_rows.append(pd.DataFrame({
            "train_hotel": source_hotel, "test_hotel": evaluation_hotel,
            "test_row_id": X_hotel_test.index.to_numpy(),
            "actual": y_hotel_test.to_numpy(), "probability": hotel_test_prob,
            "predicted": (hotel_test_prob >= source_experiment["threshold"]).astype(int),
            "Threshold": source_experiment["threshold"],
            "baseline_predicted": baseline_prediction,
            "baseline_probability": baseline_probability,
        }))
        model_metrics = probability_metrics(
            y_hotel_test, hotel_test_prob, source_experiment["threshold"]
        )
        rq2_baseline_rows.append({
            **shared_metadata, "Model": "Majority Class Baseline",
            "Baseline_class": int(baseline_prediction[0]), **baseline_metrics,
        })
        rq2_rows.append({
            **shared_metadata,
            "n_features": X_source_train.shape[1],
            "max_depth": source_experiment["max_depth"],
            "min_samples_leaf": source_experiment["min_samples_leaf"],
            "Threshold": source_experiment["threshold"], **model_metrics,
            "Baseline_ROC_AUC": baseline_metrics["ROC_AUC"],
            "ROC_AUC_above_baseline": model_metrics["ROC_AUC"] - baseline_metrics["ROC_AUC"],
        })

rq2_predictions = pd.concat(rq2_prediction_rows, ignore_index=True)
rq2_predictions.to_csv(output_dir / "decision_tree_rq2_predictions.csv", index=False)
rq2_test_results = pd.DataFrame(rq2_rows)
rq2_baseline_results = pd.DataFrame(rq2_baseline_rows)
rq2_baseline_comparison = pd.concat([
    rq2_test_results.assign(Model="Decision Tree"), rq2_baseline_results
], ignore_index=True).sort_values(["Train_hotel", "Test_hotel", "Model"])
rq2_test_results.to_csv(output_dir / "decision_tree_rq2_results.csv", index=False)
rq2_baseline_results.to_csv(output_dir / "decision_tree_rq2_baseline_results.csv", index=False)
rq2_baseline_comparison.to_csv(output_dir / "decision_tree_rq2_vs_baseline.csv", index=False)
print("RQ2 final test results (primary metric: ROC-AUC):")
display(rq2_test_results.round(4))
print("RQ2 comparison with source-trained majority baselines:")
display(rq2_baseline_comparison.round(4))
print("ROC-AUC by source and evaluation hotel:")
display(rq2_test_results.pivot(index="Train_hotel", columns="Test_hotel", values="ROC_AUC").round(4))

In [ ]:
cm = confusion_matrix(y_test, test_pred, labels=[0, 1])
pd.DataFrame(
    cm, index=["Actual_not_canceled", "Actual_canceled"],
    columns=["Predicted_not_canceled", "Predicted_canceled"],
).to_csv(output_dir / "decision_tree_confusion_matrix.csv")

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["Not canceled", "Canceled"]
).plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title(f"Decision Tree: Test Confusion Matrix\nThreshold = {final_threshold:.2f}")
fig.tight_layout()
fig.savefig(output_dir / "decision_tree_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
dt_error_analysis = X_test.copy()
dt_error_analysis.insert(0, "test_row_id", np.arange(len(X_test)))
dt_error_analysis["actual"] = y_test.to_numpy()
dt_error_analysis["probability_canceled"] = test_prob
# Alias and confusion-matrix labels match the shared LR export format.
dt_error_analysis["cancel_probability"] = dt_error_analysis["probability_canceled"]
dt_error_analysis["predicted"] = test_pred
dt_error_analysis["error_type"] = np.select([
    (dt_error_analysis["actual"] == 0) & (dt_error_analysis["predicted"] == 0),
    (dt_error_analysis["actual"] == 0) & (dt_error_analysis["predicted"] == 1),
    (dt_error_analysis["actual"] == 1) & (dt_error_analysis["predicted"] == 0),
    (dt_error_analysis["actual"] == 1) & (dt_error_analysis["predicted"] == 1),
], ["TN", "FP", "FN", "TP"], default="Unknown")
dt_error_analysis["is_error"] = dt_error_analysis["actual"] != dt_error_analysis["predicted"]
dt_error_analysis["has_booking_history"] = dt_error_analysis[history_cols].gt(0).any(axis=1)

dt_false_negatives = dt_error_analysis.loc[
    (dt_error_analysis["actual"] == 1) & (dt_error_analysis["predicted"] == 0)
].sort_values(["probability_canceled", "test_row_id"], ascending=[True, True])
dt_false_positives = dt_error_analysis.loc[
    (dt_error_analysis["actual"] == 0) & (dt_error_analysis["predicted"] == 1)
].sort_values(["probability_canceled", "test_row_id"], ascending=[False, True])
# Put LR's shared example columns first without discarding any DT evidence.
example_prefix = [
    "hotel", "lead_time", "total_nights", "total_guests", "deposit_type",
    "market_segment", "customer_type", "previous_cancellations",
    "previous_bookings_not_canceled", "cancel_probability", "actual", "predicted",
    "test_row_id",
]
example_columns = example_prefix + [
    column for column in dt_error_analysis.columns if column not in example_prefix
]
fn_examples = dt_false_negatives.head(20).loc[:, example_columns].copy()
fp_examples = dt_false_positives.head(20).loc[:, example_columns].copy()
print("Test false negatives, lowest cancellation probability first:")
display(fn_examples.head(5))
print("Test false positives, highest cancellation probability first:")
display(fp_examples.head(5))

dt_history_error_summary = (
    dt_error_analysis.groupby("has_booking_history", as_index=False)
    .agg(bookings=("is_error", "size"), errors=("is_error", "sum"),
         error_rate=("is_error", "mean"), cancellation_rate=("actual", "mean"))
)
print("Test errors by booking history (any of the three history features > 0):")
display(dt_history_error_summary.round(4))

numeric_error_features = [
    "lead_time", "total_nights", "total_guests",
    "previous_cancellations", "previous_bookings_not_canceled",
]
dt_error_summary = (
    dt_error_analysis.groupby("error_type")[numeric_error_features].mean().round(2)
)
print("Mean numeric features by error type (shared LR summary format):")
display(dt_error_summary)
dt_error_summary.to_csv(output_dir / "decision_tree_error_summary.csv")

dt_error_analysis.to_csv(output_dir / "decision_tree_error_analysis.csv", index=False)
dt_history_error_summary.to_csv(output_dir / "decision_tree_history_error_summary.csv", index=False)
fn_examples.to_csv(output_dir / "decision_tree_FN_examples.csv", index=False)
fp_examples.to_csv(output_dir / "decision_tree_FP_examples.csv", index=False)

# Audit one FP and one FN using the frozen tree and its training observations.
selected_errors = pd.concat([
    fp_examples.head(1).assign(Error_type="FP"),
    fn_examples.head(1).assign(Error_type="FN"),
], ignore_index=True)
encoded_feature_names = preprocessor.get_feature_names_out()
train_leaf_ids = best_dt.apply(preprocessor.transform(X_train))
train_labels = y_train.to_numpy()
case_rows, path_rows = [], []
for _, example in selected_errors.iterrows():
    row_id = int(example["test_row_id"])
    encoded_row = preprocessor.transform(X_test.iloc[[row_id]])
    leaf_id = int(best_dt.apply(encoded_row)[0])
    leaf_labels = train_labels[train_leaf_ids == leaf_id]
    leaf_n0 = int((leaf_labels == 0).sum())
    leaf_n1 = int((leaf_labels == 1).sum())
    leaf_probability = leaf_n1 / len(leaf_labels)
    assert np.isclose(leaf_probability, example["probability_canceled"])
    path = best_dt.decision_path(encoded_row)
    node_ids = path.indices[path.indptr[0]:path.indptr[1]]
    for step, node_id in enumerate(node_ids):
        node_id = int(node_id)
        if node_id == leaf_id:
            continue
        feature_id = int(best_dt.tree_.feature[node_id])
        value = float(encoded_row[0, feature_id])
        split_value = float(best_dt.tree_.threshold[node_id])
        # sklearn traverses trees with float32 feature values.
        takes_left = np.float32(value) <= split_value
        next_node = int(best_dt.tree_.children_left[node_id] if takes_left
                        else best_dt.tree_.children_right[node_id])
        path_rows.append({
            "test_row_id": row_id, "Error_type": example["Error_type"],
            "step": step, "node_id": node_id,
            "encoded_feature": encoded_feature_names[feature_id],
            "feature_value": value, "split_threshold": split_value,
            "taken_condition": "<=" if takes_left else ">",
            "next_node_id": next_node,
            "node_train_samples": int(best_dt.tree_.n_node_samples[node_id]),
        })
    case_rows.append({
        **example.to_dict(), "Threshold": final_threshold, "leaf_id": leaf_id,
        "leaf_train_samples": len(leaf_labels),
        "leaf_train_not_canceled": leaf_n0, "leaf_train_canceled": leaf_n1,
        "leaf_train_cancellation_rate": leaf_probability,
        "decision_path_length": len(node_ids) - 1,
    })

case_evidence = pd.DataFrame(case_rows)
case_paths = pd.DataFrame(path_rows, columns=[
    "test_row_id", "Error_type", "step", "node_id", "encoded_feature",
    "feature_value", "split_threshold", "taken_condition", "next_node_id",
    "node_train_samples",
])
case_evidence.to_csv(output_dir / "decision_tree_error_case_evidence.csv", index=False)
case_paths.to_csv(output_dir / "decision_tree_error_case_paths.csv", index=False)
print("Selected error cases: actual labels, predictions, and training evidence in the reached leaf:")
if not case_evidence.empty:
    display(case_evidence[[
        "test_row_id", "Error_type", "actual", "predicted", "probability_canceled",
        "Threshold", "leaf_id", "leaf_train_samples", "leaf_train_not_canceled",
        "leaf_train_canceled", "leaf_train_cancellation_rate",
    ]].round(4))
    print("The leaf's training cancellation rate is the model probability; compare it with the frozen threshold.")
    print("Actual test labels disagree for these selected cases. The path explains model decisions, not guest motives.")
    print("Exact model decision paths (categorical features are one-hot encoded):")
    display(case_paths)
else:
    print("There are no misclassified test cases for this model.")